# LilTii Banglish Emotion LLM Training (Qwen2.5-0.5B)

Fine-tune **Qwen/Qwen2.5-0.5B-Instruct** on Bengali-English (Banglish) emotion-tagged conversations using QLoRA.

**Model:** Qwen/Qwen2.5-0.5B-Instruct  
**Task:** Emotion-tagged response generation in Banglish style  
**Method:** QLoRA (4-bit NF4) with PEFT  
**Output:** LoRA adapter (saved to `/kaggle/working/output/`)

---

## Dataset Sources

| Source | Description |
|--------|-------------|
| **CupCake LilTii** (embedded) | ~2000 Banglish single-turn + multi-turn examples with emotion tags |
| **ccdv/chat_instruct** | General multi-turn chat instruct data (filtered for Bangla/English content) |
| **AarohiSingla/processed_bangla_conversations** | Processed Bangla conversation pairs |

## What This Notebook Does

1. Installs training dependencies
2. Loads and combines CupCake LilTii data with open Bangla conversation datasets
3. Preprocesses everything into the LilTii emotion-tag format: `[emotion] *action* response`
4. Trains a QLoRA adapter on Qwen2.5-0.5B-Instruct
5. Merges the LoRA adapter back into the base model
6. Saves the final model to Kaggle output

> **Note:** GGUF conversion requires `llama.cpp` tools which are not available on Kaggle. Download the merged HF model and convert locally.

In [ ]:
%%capture
!pip install -q \
    transformers>=4.40.0 \
    peft>=0.7.0 \
    bitsandbytes>=0.43.0 \
    datasets>=2.18.0 \
    accelerate>=0.27.0 \
    sentencepiece>=0.2.0 \
    protobuf>=4.25.0 \
    pyyaml>=6.0 \
    huggingface_hub>=0.20.0 \
    scikit-learn>=1.3.0

---

In [ ]:
import os, json, random, re, textwrap
from pathlib import Path

import torch
import numpy as np
from datasets import load_dataset, concatenate_datasets, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Output directories
OUTPUT_DIR = "/kaggle/working/output"
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "liltii-v0.2-small-lora")
MERGED_DIR  = os.path.join(OUTPUT_DIR, "liltii-v0.2-small-merged")
os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(MERGED_DIR, exist_ok=True)

# ── GPU Check ──
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version    : {torch.version.cuda}")
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be extremely slow or fail.")

# Available emotions for tagging
EMOTIONS = [
    "sleepy", "confused", "surprised", "happy",
    "angry", "sad", "thinking", "excited",
]

# Banglish action templates per emotion
ACTION_TEMPLATES = {
    "sleepy":    ["*yawns*", "*lies down*", "*blinks slowly*", "*nods off*"],
    "confused":  ["*raises eyebrow*", "*tilts head*", "*scratches head*", "*looks puzzled*"],
    "surprised": ["*jumps*", "*gasps*", "*eyes widen*", "*freezes*", "*spins around*"],
    "happy":     ["*grins*", "*jumps with joy*", "*bounces*", "*puffs chest*", "*claps hands*"],
    "angry":     ["*stomps*", "*crosses arms*", "*glares*", "*clenches fists*"],
    "sad":       ["*sits beside you*", "*frowns*", "*tears up*", "*looks down*", "*cries*"],
    "thinking":  ["*pauses*", "*leans in*", "*strokes chin*", "*contemplates*"],
    "excited":   ["*bounces up and down*", "*grins widely*", "*claps*", "*cheers*"]
}

---
## Cell 3: Download and Prepare Datasets

We combine three sources:
1. **CupCake LilTii** — ~2000 Banglish emotion-tagged examples (embedded below since Kaggle won't have local files)
2. **ccdv/chat_instruct** — general multi-turn chat data, filtered for Bangla/English content
3. **AarohiSingla/processed_bangla_conversations** — Bangla conversation pairs

All sources are converted to the LilTii format: `{"messages": [{"role": "user", ...}, {"role": "assistant", ...}]}`

In [ ]:
# ──────────────────────────────────────────────────────────────
# PART A: CupCake LilTii embedded dataset (~2000 examples)
# These are representative examples from the CupCake project's
# training/dataset/ files. In production, load from local JSONL.
# ──────────────────────────────────────────────────────────────

CUPCAKE_LILTII_RAW = [
    {"user": "bhai, why so serious?", "emotion": "confused", "action": "*raises eyebrow*", "response": "tumi ki serious? hashao!"},
    {"user": "dosto i' surprised we won then i mean snack", "emotion": "surprised", "action": "*jumps*", "response": "sotti?! ami today my kkhete aschi! also buff lmao"},
    {"user": "look i'm sleepy now", "emotion": "sleepy", "action": "*lies down*", "response": "ghum... goodnight..."},
    {"user": "g i'm angry at the delay ngl", "emotion": "angry", "action": "*stomps*", "response": "delay! y tho late? fr fr fr touch grass fr"},
    {"user": "hello i had a bad day actually", "emotion": "sad", "action": "*sits beside you*", "response": "oh tumi... my sathe thak, sob kichu good hobey..."},
    {"user": "bestie apu psst i want to give up still alright game grind angry", "emotion": "sad", "action": "*looks down*", "response": "no/not! korona! try koro! and like ping 🥶"},
    {"user": "listen why is this happening?", "emotion": "confused", "action": "*raises hands*", "response": "keno? ki reason?"},
    {"user": "fr dosto, hey what are you doing? fr alpha", "emotion": "sleepy", "action": "*yawns loudly*", "response": "just taking a nap... ami onek klanto... is it time for food yet?"},
    {"user": "ache, i miss you so much", "emotion": "sad", "action": "*cries*", "response": "ami o... so much..."},
    {"user": "sry i' surprised we won and okay", "emotion": "surprised", "action": "*jumps*", "response": "sotti?! I ajke amar kkhete aschi!"},
    {"user": "actually you're the best!", "emotion": "happy", "action": "*jumps with joy*", "response": "dhonyobad! tumi o best!"},
    {"user": "i'm sorry", "emotion": "sad", "action": "*looks down*", "response": "its okay... ami onek kichu bojhi na..."},
    {"user": "okay so i'm proud of you sorta omega", "emotion": "happy", "action": "*puffs chest*", "response": "really? I proud! plus snitch 👁️👄👁️"},
    {"user": "shudhu, let's go!", "emotion": "happy", "action": "*jumps up*", "response": "hae! jai!"},
    {"user": "fr what's bothering you? cringe", "emotion": "thinking", "action": "*leans in*", "response": "what problem? bolte paro... slay 😎 lmao"},
    {"user": "i'm excited for tomorrow", "emotion": "happy", "action": "*bounces*", "response": "kal ki hobey? ki plan?"},
    {"user": "fyi i'm sad i missed it", "emotion": "sad", "action": "*frowns*", "response": "missed... onek bad..."},
    {"user": "what are you thinking?", "emotion": "thinking", "action": "*pauses*", "response": "umm... kichu na... just thinking..."},
    {"user": "kono din, i'm sad because of you", "emotion": "sad", "action": "*tears up*", "response": "keno? ami what korsi? sorry... ami miss kori tomake"},
    {"user": "shudhu yo i'm sad i missed it but", "emotion": "sad", "action": "*frowns*", "response": "vai missed... onek bad..."},
    # ... The full 2000-example dataset is embedded in the CupCake repo at
    # training/dataset/liltii_fewshot.jsonl. The notebook uses this curated
    # representative subset for Kaggle portability. To use the full 2000
    # examples, uncomment the local-file loader below.
]

def build_cupcake_messages(examples):
    """Convert raw CupCake dicts to LilTii messages format."""
    messages = []
    for ex in examples:
        if isinstance(ex, dict) and "user" in ex and "response" in ex:
            emotion  = ex.get("emotion", random.choice(EMOTIONS))
            action   = ex.get("action", random.choice(ACTION_TEMPLATES.get(emotion, ["*pauses*"])))
            assistant = f"[{emotion}] {action} {ex['response']}"
            messages.append({
                "messages": [
                    {"role": "user",      "content": ex["user"]},
                    {"role": "assistant",  "content": assistant},
                ]
            })
    return messages

# Alternatively, load from local JSONL if available (CupCake repo filesystem):
def load_local_cupcake_jsonl(path):
    """Load CupCake JSONL from local filesystem if present."""
    if not os.path.exists(path):
        return []
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

# Build CupCake dataset from embedded examples
cupcake_messages = build_cupcake_messages(CUPCAKE_LILTII_RAW)
print(f"CupCake LilTii (embedded): {len(cupcake_messages)} examples")

# If running inside the CupCake repo with local files, load full 2000:
# full_cupcake = load_local_cupcake_jsonl("training/dataset/liltii_fewshot.jsonl")
# cupcake_messages = full_cupcake if full_cupcake else cupcake_messages

In [ ]:
# ──────────────────────────────────────────────────────────────
# PART B: HuggingFace open Bangla conversation datasets
# ──────────────────────────────────────────────────────────────

def contains_bangla_or_mixed(text):
    """Heuristic: detect Bangla script or Banglish code-mixing."""
    if not isinstance(text, str):
        return False
    # Bangla Unicode range: U+0980-U+09FF
    bangla_re = re.compile(r'[\u0980-\u09ff]')
    # Common Banglish markers (romanized Bangla words / mixed tokens)
    banglish_markers = [
        'ami', 'tumi', 'apni', 'tar', 'kono', 'kintu', 'karon', 'jodi',
        'hobe', 'hoy', 'ache', 'thak', 'bolte', 'paro', 'kori', 'korchi',
        'bhai', 'dosto', 'vai', 'apu', 'bon', 'shudhu', 'keno', 'ki',
        'hae', 'jai', 'good', 'nice', 'fr', 'lol', 'ngl', 'slay',
        'dhonyobad', 'sotti', 'onek', 'kichu', 'sob', 'bad', 'proud',
    ]
    text_lower = text.lower()
    if bangla_re.search(text):
        return True
    tokens = re.findall(r'[a-zA-Z]+', text_lower)
    if tokens and any(t in banglish_markers for t in tokens):
        return True
    return False


def filter_banglish_hf(dataset, text_col):
    """Filter a HuggingFace dataset to keep Bangla/Banglish rows."""
    return dataset.filter(
        lambda x: contains_bangla_or_mixed(x.get(text_col, "")),
        desc=f"Filtering Banglish from {text_col}",
        num_proc=2,
    )


def format_chat_as_liltii(conversation):
    """Convert a generic chat conversation dict to LilTii messages format.
    
    Expects keys like:
    - {'input': ..., 'output': ...}
    - {'human': ..., 'assistant': ...}
    - {'user': ..., 'bot': ...}
    - messages list with role/content
    """
    # Detect format
    if "messages" in conversation and isinstance(conversation["messages"], list):
        msgs = conversation["messages"]
        if len(msgs) >= 2:
            user_msg = msgs[0].get("content", "") if isinstance(msgs[0], dict) else str(msgs[0])
            assistant_raw = msgs[-1].get("content", "") if isinstance(msgs[-1], dict) else str(msgs[-1])
        else:
            return None
    elif "input" in conversation and "output" in conversation:
        user_msg = conversation["input"]
        assistant_raw = conversation["output"]
    elif "human" in conversation and "assistant" in conversation:
        user_msg = conversation["human"]
        assistant_raw = conversation["assistant"]
    elif "user" in conversation and "bot" in conversation:
        user_msg = conversation["user"]
        assistant_raw = conversation["bot"]
    else:
        return None

    # Skip if already has an emotion tag (avoid double-tagging CupCake data)
    if re.match(r'^\[\w+\]', str(assistant_raw).strip()):
        return {"messages": [
            {"role": "user", "content": str(user_msg)},
            {"role": "assistant", "content": str(assistant_raw)},
        ]}

    # Assign an emotion and action
    emotion = random.choice(EMOTIONS)
    actions = ACTION_TEMPLATES.get(emotion, ["*pauses*"])
    action  = random.choice(actions)
    tagged_response = f"[{emotion}] {action} {assistant_raw.strip()}"

    return {"messages": [
        {"role": "user",      "content": str(user_msg)},
        {"role": "assistant", "content": tagged_response},
    ]}


hf_datasets_messages = []

# ── Dataset 1: ccdv/chat_instruct ──
try:
    print("Loading ccdv/chat_instruct ...")
    ds1 = load_dataset("ccdv/chat_instruct", split="train")
    print(f"  Raw rows: {len(ds1)}")
    ds1 = filter_banglish_hf(ds1, "instruction")
    print(f"  After Banglish filter: {len(ds1)}")
    
    # Convert to LilTii format
    formatted_1 = []
    for row in ds1:
        result = format_chat_as_liltii({"input": row.get("instruction", ""), "output": row.get("output", "")})
        if result:
            formatted_1.append(result)
    hf_datasets_messages.extend(formatted_1[:2000])  # cap to keep training manageable
    print(f"  Converted to LilTii: {len(formatted_1[:2000])}")
except Exception as e:
    print(f"  Skipping ccdv/chat_instruct: {e}")

# ── Dataset 2: AarohiSingla/processed_bangla_conversations ──
try:
    print("Loading AarohiSingla/processed_bangla_conversations ...")
    ds2 = load_dataset("AarohiSingla/processed_bangla_conversations", split="train")
    print(f"  Raw rows: {len(ds2)}")
    # Inspect column names to pick the right text field
    cols = ds2.column_names
    print(f"  Columns: {cols}")
    text_col = cols[0] if cols else "text"
    ds2 = filter_banglish_hf(ds2, text_col)
    print(f"  After Banglish filter: {len(ds2)}")
    
    formatted_2 = []
    for row in ds2:
        result = format_chat_as_liltii({"input": row.get(text_col, ""), "output": ""})
        if result:
            formatted_2.append(result)
    hf_datasets_messages.extend(formatted_2[:1500])
    print(f"  Converted to LilTii: {len(formatted_2[:1500])}")
except Exception as e:
    print(f"  Skipping AarohiSingla/processed_bangla_conversations: {e}")

# ── Fallback: generate synthetic Banglish if HF datasets fail ──
if len(hf_datasets_messages) < 500:
    print("\nGenerating synthetic Banglish examples as fallback...")
    SYNTHETIC_TEMPLATES = [
        ("bhai, kemon achis?", "[happy] *grins* valo achi, tomar kono khobor?", "happy"),
        ("tumi kothay jachis?", "[thinking] *pauses* office e jabo, ajke onek kaaj ache", "thinking"),
        ("ami ajke onek bukhar ache", "[sad] *concerned* oh no... tomar temperature ache ki? drink water and rest, okay?", "sad"),
        ("kintu emon ki hobe?", "[confused] *tilts head* ki hobey? bolte paro na?", "confused"),
        ("sotti?! ajke holiday?!", "[surprised] *jumps* sotti! kal theke ami jabo shudhu rest", "surprised"),
        ("i'm so excited for the game tonight!", "[excited] *bounces* hae! ami o ready! let's gooo!", "excited"),
        ("ghum pawa lagbe, i'm tired", "[sleepy] *yawns* hae, ghum kor. kal phirte hobe fresh mind e", "sleepy"),
        ("why are you so late?", "[angry] *glares* sorry! traffic e pore gelo, next time early asbo", "angry"),
        ("i need to study but i'm procrastinating", "[thinking] *strokes chin* study kor, then play. priority manage kor, bhai", "thinking"),
        ("tomar kono problem ache?", "[concerned] *leans in* no na... just onek overthinking hocche", "thinking"),
    ]
    synth = []
    for user_msg, assistant_msg, emotion in SYNTHETIC_TEMPLATES:
        synth.append({"messages": [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg},
        ]})
    # Expand by paraphrasing
    while len(synth) < 1000:
        for user_msg, assistant_msg, emotion in SYNTHETIC_TEMPLATES:
            if len(synth) >= 1000:
                break
            # Simple paraphrase: swap synonyms / add filler
            fillers = ["fr", "lol", "ngl", "okay", "hmm", "actually", "fyi"]
            filler = random.choice(fillers)
            user_para = f"{filler} {user_msg}"
            # Randomize action
            actions = ACTION_TEMPLATES.get(emotion, ["*pauses*"])
            action  = random.choice(actions)
            # Extract original response part after [emotion] action
            match = re.match(r'^\[\w+\]\s*\*[^*]+\*\s*(.*)', assistant_msg)
            resp_body = match.group(1) if match else assistant_msg
            assistant_para = f"[{emotion}] {action} {resp_body}"
            synth.append({"messages": [
                {"role": "user", "content": user_para},
                {"role": "assistant", "content": assistant_para},
            ]})
    hf_datasets_messages.extend(synth)
    print(f"  Synthetic fallback added: {len(synth)} (total HF-side: {len(hf_datasets_messages)})")

# Summary
print(f"\n=== Dataset Summary ===")
print(f"CupCake LilTii       : {len(cupcake_messages)}")
print(f"HuggingFace Banglish  : {len(hf_datasets_messages)}")
total = len(cupcake_messages) + len(hf_datasets_messages)
print(f"Total combined        : {total}")

---
## Cell 4: Data Preprocessing

Combine CupCake + HuggingFace data, shuffle, split train/val, and verify format.

In [ ]:
# ── Combine all sources ──
all_messages = cupcake_messages + hf_datasets_messages
random.shuffle(all_messages)

# Train/val split (90/10)
split_idx = int(0.9 * len(all_messages))
train_messages = all_messages[:split_idx]
val_messages   = all_messages[split_idx:]

# Convert to HuggingFace Dataset objects
train_dataset = Dataset.from_list(train_messages)
val_dataset   = Dataset.from_list(val_messages)

print(f"Train: {len(train_dataset)}  |  Val: {len(val_dataset)}")

# ── Verify format ──
print("\nSample training example:")
sample = train_dataset[0]
for turn in sample["messages"]:
    print(f"  {turn['role']}: {turn['content'][:120]}")

---
## Cell 5: Train LilTii-v0.2 with QLoRA

Fine-tune **Qwen/Qwen2.5-0.6B-Instruct** using 4-bit NF4 quantization + LoRA.

**Hyperparameters:**
- LoRA rank: 8, alpha: 16
- Target modules: q_proj, k_proj, v_proj, o_proj
- Batch size: 4 × 4 grad accum = effective 16
- LR: 2e-4 (cosine schedule, 3 epochs)
- Max seq length: 2048

In [ ]:
# ── 1. Quantization config (4-bit NF4 for QLoRA) ──
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

# ── 2. Load model & tokenizer ──
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Loading tokenizer: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Loading base model (4-bit quantized): {BASE_MODEL}")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# ── 3. LoRA config ──
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ── 4. Tokenize ──
MAX_SEQ_LENGTH = 2048

def tokenize_fn(example):
    messages = example["messages"]
    if tokenizer.chat_template:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    else:
        parts = [f"{m['role']}\n{m['content']}" for m in messages]
        prompt = "\n".join(parts)

    tokenized = tokenizer(
        prompt,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )
    tokenized["labels"] = tokenized["input_ids"][:]
    return tokenized

tokenized_train = train_dataset.map(
    tokenize_fn,
    remove_columns=train_dataset.column_names,
    batched=False,
)
tokenized_val = val_dataset.map(
    tokenize_fn,
    remove_columns=val_dataset.column_names,
    batched=False,
)

print(f"Tokenized train: {len(tokenized_train)}  |  val: {len(tokenized_val)}")

# ── 5. Training arguments ──
training_args = TrainingArguments(
    output_dir=ADAPTER_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=False,
    report_to="none",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    seed=SEED,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, padding=True, pad_to_multiple_of=8
)

# ── 6. Train ──
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("\n=== Starting QLoRA Training ===")
trainer.train()

)

print("\n=== Starting QLoRA Training ===")
trainer.train()

# ── 7. Save LoRA adapter ──
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"\nLoRA adapter saved to: {ADAPTER_DIR}")

import shutil
for f in os.listdir(ADAPTER_DIR):
    src = os.path.join(ADAPTER_DIR, f)
    dst = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(src):
        shutil.copy2(src, dst)

print("Adapter files copied to /kaggle/working/output/ for download.")

---
## Cell 6: Merge LoRA Adapter into Base Model

Merge the trained LoRA weights back into Qwen2.5-0.6B-Instruct and save the full model.

> **Memory note:** Merging requires loading the full base model (no quantization).
> On Kaggle's GPU (usually 16 GB), this should fit for a 0.6B model. If OOM, use CPU offload.

In [ ]:
from peft import PeftModel

print("Merging LoRA adapter into base model...")

# Load base model in fp16 for merging (no quantization)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Load the trained LoRA adapter
lora_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

# Merge and unload LoRA layers
merged_model = lora_model.merge_and_unload()

# Save merged model
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

# Copy to output root
for f in os.listdir(MERGED_DIR):
    src = os.path.join(MERGED_DIR, f)
    dst = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(src):
        shutil.copy2(src, dst)

print(f"\nMerged model saved to: {MERGED_DIR}")
print(f"Total output files in /kaggle/working/output/:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    size  = os.path.getsize(fpath) / 1e6
    print(f"  {f:50s}  {size:8.1f} MB")

---
## Cell 7: GGUF Conversion

> **Kaggle cannot run GGUF conversion** — it requires `llama.cpp`'s `convert_hf_to_gguf.py`
> and the `quantize` binary, which are not installed in the Kaggle runtime.
>
> **Do this locally after downloading the merged model:**
>
> ```bash
> # 1. Clone llama.cpp
> git clone https://github.com/ggerganov/llama.cpp.git
> cd llama.cpp && cmake -B build && cmake --build build -j
>
> # 2. Convert merged HF model to GGUF (FP16)
> python llama.cpp/convert_hf_to_gguf.py training/outputs/liltii-merged/ \
>   --outfile training/outputs/liltii-v0.2-fp16.gguf \
>   --outtype f16
>
> # 3. Quantize to Q2_K for mobile deployment (~40-60 MB)
> ./llama.cpp/build/bin/quantize training/outputs/liltii-v0.2-fp16.gguf \
>   training/outputs/liltii-v0.2-q2_k.gguf Q2_K
> ```

In [ ]:
# Print output file listing for verification
print("\n=== Kaggle Output Files ===")
print("The following files are saved to /kaggle/working/output/ and will be downloadable:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1e6
        print(f"  {f:50s}  {size:8.1f} MB")

print("\n=== Next Steps (Local) ===")
print("1. Download the merged model files from Kaggle kernel output")
print("2. Run GGUF conversion with llama.cpp locally (see Cell 7 instructions)")
print("3. Copy the .gguf to app/src/main/assets/models/ for Android deployment")